# Model Interpretability — Confusion Matrices & Grad-CAM

**Project:** CaféScan — Coffee Leaf Disease Detection  
**Purpose:** Display confusion matrices and Grad-CAM visualizations for each model to understand
where predictions succeed, where they fail, and what image regions each model attends to.

## What is Grad-CAM?

**Gradient-weighted Class Activation Mapping (Grad-CAM)** is a technique for producing
visual explanations for CNN decisions. It computes the gradient of the class score with
respect to the feature maps of the last convolutional layer, then uses those gradients
as weights to produce a coarse localization heatmap highlighting the discriminative regions.

- **Hot regions (red/yellow):** Areas the model focused on to make its prediction.
- **Cold regions (blue):** Less relevant to the classification decision.

**Note:** Grad-CAM is designed for CNN architectures. For **ViT**, which uses self-attention
rather than convolutional feature maps, Grad-CAM is not directly applicable —
attention rollout or Attention Flow methods are required instead.

**Models:** efficientnet_b0, resnet50, vit, mobilenet  
**Classes:** healthy, rust, cercospora, miner, phoma

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')

ROOT = Path(".")
RESULTS = ROOT / "results"
TABLES = RESULTS / "tables"
FIGURES = RESULTS / "figures"

MODELS = ["efficientnet_b0", "resnet50", "vit", "mobilenet"]
CNN_MODELS = ["efficientnet_b0", "resnet50", "mobilenet"]  # Grad-CAM applicable
MODEL_LABELS = {
    "efficientnet_b0": "EfficientNet-B0",
    "resnet50": "ResNet-50",
    "vit": "ViT",
    "mobilenet": "MobileNet",
}
CLASSES = ["healthy", "rust", "cercospora", "miner", "phoma"]
CLASS_LABELS = [c.title() for c in CLASSES]

cmap = plt.get_cmap("tab10")
MODEL_COLORS = {m: cmap(i) for i, m in enumerate(MODELS)}

print("Imports OK. ROOT:", ROOT.resolve())

## 1. Confusion Matrices — All 4 Models

Displayed in a 2×2 grid. Each matrix shows predicted vs true labels for the 5 classes
on the test set.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes_flat = axes.flatten()

for idx, model in enumerate(MODELS):
    cm_path = RESULTS / model / "confusion_matrix.png"
    ax = axes_flat[idx]
    try:
        img = plt.imread(str(cm_path))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(
            f"{MODEL_LABELS[model]} — Confusion Matrix",
            fontsize=12,
            fontweight="bold",
            pad=8,
            color=MODEL_COLORS[model],
        )
    except FileNotFoundError:
        ax.text(0.5, 0.5, f"File not found:\n{cm_path}",
                ha="center", va="center", fontsize=10, color="red")
        ax.axis("off")

fig.suptitle(
    "Confusion Matrices — Test Set (All Models)",
    fontsize=15, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig(FIGURES / "nb_confusion_matrices_grid.png", dpi=120, bbox_inches="tight")
plt.show()
plt.close()

## 2. Grad-CAM Visualizations — CNN Models

Each row = one CNN model. Each column = one disease class.  
Grad-CAM maps highlight which image regions the model focused on when classifying each disease.

In [ ]:
n_rows = len(CNN_MODELS)
n_cols = len(CLASSES)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 10))

for row_idx, model in enumerate(CNN_MODELS):
    for col_idx, cls in enumerate(CLASSES):
        gradcam_path = RESULTS / model / f"gradcam_{cls}.png"
        ax = axes[row_idx, col_idx]
        try:
            img = plt.imread(str(gradcam_path))
            ax.imshow(img)
            ax.axis("off")
            # Column header (class name) only on first row
            if row_idx == 0:
                ax.set_title(cls.title(), fontsize=11, fontweight="bold")
        except FileNotFoundError:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center",
                    fontsize=9, color="gray")
            ax.axis("off")
            if row_idx == 0:
                ax.set_title(cls.title(), fontsize=11, fontweight="bold")

    # Row label (model name) on y-axis of first column
    axes[row_idx, 0].set_ylabel(
        MODEL_LABELS[model],
        fontsize=11,
        fontweight="bold",
        rotation=90,
        labelpad=8,
        color=MODEL_COLORS[model],
    )
    axes[row_idx, 0].yaxis.set_visible(True)

fig.suptitle(
    "Grad-CAM Visualizations — CNN Models × Disease Classes",
    fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig(FIGURES / "nb_gradcam_grid.png", dpi=120, bbox_inches="tight")
plt.show()
plt.close()

## 3. ViT — Grad-CAM Not Applicable

> **Note:** Vision Transformer (ViT) does not use convolutional feature maps, so standard
> Grad-CAM cannot be applied. ViT processes images as sequences of patch embeddings and
> uses multi-head self-attention to relate patches globally.
>
> To interpret ViT decisions, the appropriate techniques are:
> - **Attention Rollout** — recursively multiplies attention matrices across layers to trace
>   which input patches most influenced the CLS token used for classification.
> - **Attention Flow** — tracks information flow through the attention graph.
> - **Generic Grad-CAM variants** (e.g., Grad-CAM++ on the last MLP layer) — less
>   interpretable than CNN equivalents.
>
> Despite lacking Grad-CAM, ViT achieved **99.65% Macro-F1** on the test set — the best
> performance among all 4 models, demonstrating that its global attention mechanism
> captures disease-discriminative features very effectively.

## 4. Per-Class F1 Analysis — Hardest Class per Model

In [ ]:
raw_df = pd.read_csv(TABLES / "test_summary.csv").set_index("model")
f1_cols = [f"f1_{cls}" for cls in CLASSES]
f1_data = raw_df[f1_cols].copy()
f1_data.index = [MODEL_LABELS.get(m, m) for m in f1_data.index]

# Identify hardest class per model
print("Hardest class (lowest F1) per model:")
print("-" * 45)
for model_label in f1_data.index:
    row = f1_data.loc[model_label]
    hardest_col = row.idxmin()
    hardest_cls = hardest_col.replace("f1_", "").title()
    score = row[hardest_col]
    print(f"  {model_label:20s} → {hardest_cls:12s} (F1 = {score:.4f})")

print("\nEasiest class (highest F1) per model:")
print("-" * 45)
for model_label in f1_data.index:
    row = f1_data.loc[model_label]
    easiest_col = row.idxmax()
    easiest_cls = easiest_col.replace("f1_", "").title()
    score = row[easiest_col]
    print(f"  {model_label:20s} → {easiest_cls:12s} (F1 = {score:.4f})")

In [ ]:
# Bar chart: per-class F1 for all models, grouped by class
n_models = len(f1_data)
n_classes = len(CLASSES)
x = np.arange(n_classes)
width = 0.18
offsets = np.linspace(-(n_models - 1) / 2, (n_models - 1) / 2, n_models) * width

fig, ax = plt.subplots(figsize=(13, 6))

for i, model_label in enumerate(f1_data.index):
    original_key = [k for k, v in MODEL_LABELS.items() if v == model_label]
    color = MODEL_COLORS[original_key[0]] if original_key else cmap(i)
    vals = [f1_data.loc[model_label, f"f1_{cls}"] for cls in CLASSES]
    bars = ax.bar(x + offsets[i], vals, width, label=model_label, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_LABELS, fontsize=12)
ax.set_ylabel("F1 Score", fontsize=12)
ax.set_title(
    "Per-Class F1 Scores — Test Set (All Models)",
    fontsize=13, fontweight="bold"
)
ax.set_ylim(0.5, 1.05)
ax.legend(fontsize=10, loc="lower right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.2f}"))

# Reference line at 0.90
ax.axhline(y=0.90, color="gray", linestyle="--", alpha=0.5, linewidth=1, label="F1=0.90")

plt.tight_layout()
plt.savefig(FIGURES / "nb_per_class_f1_bar.png", dpi=120, bbox_inches="tight")
plt.show()
plt.close()

## 5. Grad-CAM Interpretation — What the Maps Reveal

### EfficientNet-B0
EfficientNet's compound-scaled architecture, with its wider and deeper feature extraction,
produces Grad-CAM maps that tend to focus on **texture regions** — the specific lesion
patterns, discoloration patches, and surface anomalies that characterize each disease.
Given its 95.40% test Macro-F1, the activation maps should be tightly localized around
the disease-affected leaf tissue.

- **Rust:** Expect activation on orange-brown pustule clusters on the leaf underside.
- **Cercospora:** Attention on circular necrotic spots with yellow halos.
- **Miner:** Focus on serpentine mines and blister-like surface damage.
- **Phoma:** Activation on dark, watersoaked lesion margins.
- **Healthy:** Diffuse, unfocused activations — no single discriminating region.

### ResNet-50
ResNet-50 shows lower test performance (80.39%), which may manifest in Grad-CAM maps
as **diffuse or mislocalized activations** — the model may attend to background elements
or non-discriminative leaf regions, suggesting the residual blocks alone are insufficient
for fine-grained disease texture discrimination.

### MobileNet
MobileNet's depthwise separable convolutions reduce parameter count but can miss subtle
textural details. Grad-CAM maps may show **coarser, less precise localization**
compared to EfficientNet. At 91.38% test Macro-F1, the model captures most disease patterns
but likely struggles with the visually similar classes (e.g., miner vs cercospora).

### Cross-Model Comparison
Comparing Grad-CAM maps across models reveals:
1. **Consistency check:** If all three CNN models focus on the same region, that region
   is genuinely discriminative for the disease.
2. **Shortcut learning detection:** If a model focuses on image borders or background
   instead of leaf tissue, it may have learned dataset artifacts rather than disease features.
3. **Class confusion insight:** Classes with low F1 (miner for weaker models) will show
   activation maps that overlap with the hardest confounding class.

### Practical Implication for CaféScan
For a field deployment tool used by coffee farmers, model interpretability is essential:
- Farmers and agronomists can verify that the model is attending to actual disease symptoms.
- Grad-CAM overlays can serve as visual explanations in the app, increasing trust.
- Mislocalized activations in ResNet-50 strengthen the case for using ViT or EfficientNet-B0
  in production.